# Posit Connect Inventory Scan

Discovers deployed content and stores its package inventory in PostgreSQL.

Configuration comes from environment variables set in this content item's **Vars** panel.

In [ ]:
from clients.connect_client import ConnectClient
from config.logging_config import configure_logging
from config.settings import get_settings
from database.connection import get_database
from services.inventory_service import InventoryService

configure_logging(level='INFO', log_dir=None, force=True)
settings = get_settings()
print('Connect  :', settings.connect_server_url)
print('Database :', settings.database_url_string())

## Connect and prepare the schema

In [ ]:
from database.migrations import current_revision, head_revision

database = get_database(settings)
database.verify_connection()

print('Revision before :', current_revision(database.engine))
print('Revision target :', head_revision())

# The database is only reachable from inside the VNet, so migrations
# run here rather than from a workstation. Safe on every run.
revision = database.ensure_schema()
print('Revision after  :', revision)

# Checks columns as well as tables, so a stale schema fails here
# rather than deep inside a later query.
database.verify_schema()
print('Schema verified.')

## Run the scan

In [ ]:
names = []


def on_discovered(count):
    print('Found', count, 'deployed applications.')
    print()


def on_progress(name, ok, count):
    names.append((name, ok, count))


with ConnectClient(settings) as client:
    info = client.verify_connection()
    print('Connected to Posit Connect', info['version'],
          'as', info['username'], '(role=' + str(info['user_role']) + ')')
    print()

    service = InventoryService(client, database, settings)
    result = service.run(on_discovered=on_discovered, progress=on_progress)

print()
print('Inventory Scan Complete')
print()
print('Applications Stored:', result.applications_stored)
print('Packages Stored:', result.packages_stored)

## Results

In [ ]:
for name, ok, count in names:
    mark = 'OK' if ok else '!!'
    print(mark, name, '-', count, 'packages')

if result.failures:
    print()
    print(len(result.failures), 'failure(s):')
    for guid, message in result.failures[:20]:
        print(' -', guid, ':', message[:150])

In [ ]:
from repositories.application_repository import ApplicationRepository
from repositories.package_repository import PackageRepository

with database.session() as session:
    apps = ApplicationRepository(session)
    packages = PackageRepository(session)
    print('Stored in PostgreSQL')
    print('  applications        :', apps.count())
    print('  packages            :', packages.count())
    print('  distinct owners     :', apps.distinct_owner_count())
    print('  unique pkg versions :', packages.distinct_package_versions())
    print('  by runtime          :', packages.package_type_breakdown())

database.dispose()

## CSV export for the OIS scanning team

In [ ]:
import base64
from IPython.display import HTML, display

from services.export_service import ExportService

export = ExportService(database)
csv_text = export.to_string()
filename = export.suggested_filename()

row_count = csv_text.count(chr(13) + chr(10)) - 1
size_kb = len(csv_text.encode('utf-8')) / 1024
print('Rows :', row_count)
print('Size :', round(size_kb, 1), 'KB')
print('File :', filename)

# Embedded so the link works from the rendered report, which is static HTML.
payload = base64.b64encode(csv_text.encode('utf-8-sig')).decode('ascii')
display(HTML(
    '<a download="' + filename + '" '
    'href="data:text/csv;base64,' + payload + '" '
    'style="display:inline-block;padding:10px 18px;background:#447099;'
    'color:#fff;border-radius:4px;text-decoration:none;font-weight:600">'
    'Download ' + filename + '</a>'
))

## Zip download for the DevSecOps upload

In [ ]:
# Same CSV, zipped, for portals that take an archive upload.
zip_name = export.suggested_filename(extension='zip')
zip_bytes = export.to_zip_bytes(arcname=filename)

print('File :', zip_name)
print('Size :', round(len(zip_bytes) / 1024, 1), 'KB')

zip_payload = base64.b64encode(zip_bytes).decode('ascii')
display(HTML(
    '<a download="' + zip_name + '" '
    'href="data:application/zip;base64,' + zip_payload + '" '
    'style="display:inline-block;padding:10px 18px;background:#447099;'
    'color:#fff;border-radius:4px;text-decoration:none;font-weight:600">'
    'Download ' + zip_name + '</a>'
))

In [ ]:
print(chr(10).join(csv_text.splitlines()[:6]))